# Ilera-ATLAS · 06_export: publish the chosen model (merged 16-bit + GGUF)

**Runbook Part 10 · Thu 8 Oct**, after `04_eval` picked the winning run.

| Output | Repo | Used by |
|---|---|---|
| merged 16-bit weights (~16 GB) | `<you>/ilera-atlas` | the GPU API on Modal (vLLM) |
| GGUF `Q4_K_M` (~4.9 GB) | `<you>/ilera-atlas-GGUF` | free CPU Hugging Face Space, laptops |

Both repos are created **private**; make them public on submission day. Each gets a model card with
the N-ATLaS licence terms and, if `results/compare_base_vs_ilera.csv` exists, the results table.

**Kaggle:** GPU T4 x2 · Internet On · secret `HF_TOKEN` · **Commit mode** (export takes 30–60 min).
Big files are written to `/kaggle/tmp` (or `/tmp`), not `/kaggle/working`, which is limited to ~20 GB.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"            # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"
ADAPTER = f"{HF_USER}/ilera-atlas-lora-runB"   # <- the run 04_eval chose on the validation set
MERGED_REPO = f"{HF_USER}/ilera-atlas"
GGUF_REPO = f"{HF_USER}/ilera-atlas-GGUF"
QUANT = "q4_k_m"
DO_MERGED, DO_GGUF, DO_SANITY = True, True, True
UPLOAD = True
LOCAL_DIR = None                         # offline testing of the model-card cell only

In [ ]:
import os, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
WORK = "/kaggle/tmp" if os.path.isdir("/kaggle") else "/tmp"
os.makedirs(WORK, exist_ok=True)
# If the next cells fail with an import error: Run > Restart kernel, then Run All again.

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
if UPLOAD or not LOCAL_DIR:
    assert HF_TOKEN, "No HF_TOKEN found: add it as a Kaggle/Colab secret or env var"
    os.environ["HF_TOKEN"] = HF_TOKEN

## 1. Model cards
Built first (no GPU needed) so they can be checked before the long export. The results table comes
from `04_eval` (`compare_base_vs_ilera.csv`); without it the card says results are pending.

In [ ]:
import csv, datetime

def fetch(path):
    if LOCAL_DIR:
        p = os.path.join(LOCAL_DIR, path)
        return p if os.path.exists(p) else None
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(DATA_REPO, path, repo_type="dataset", token=HF_TOKEN)
    except Exception:
        return None

def results_table():
    p = fetch("results/compare_base_vs_ilera.csv")
    if not p:
        return "_Results pending: run 04_eval with COMPARE_WITH = \"base\"._\n"
    rows = list(csv.DictReader(open(p, encoding="utf-8")))
    want = {"action_ok": "Correct ACTION", "under_referral": "Under-referral (lower is better)",
            "key_fact_recall": "Key-fact recall", "format_ok": "Answer format valid"}
    out = ["| Subset | Language | Metric | N-ATLaS (base) | Ilera-ATLAS | Difference [95% CI] |",
           "|---|---|---|---|---|---|"]
    for r in rows:
        if r["metric"] in want and (r["lang"] in ("ALL", "en", "ha", "yo")):
            sig = " *" if r["significant"] == "True" else ""
            out.append(f"| {r['subset']} | {r['lang']} | {want[r['metric']]} | {float(r['base']):.3f} | "
                       f"{float(r['ilera']):.3f} | {float(r['diff']):+.3f} [{float(r['ci_low']):+.3f}, {float(r['ci_high']):+.3f}]{sig} |")
    return "\n".join(out) + "\n\n`*` = 95% paired-bootstrap CI excludes 0. Benchmark: Ilera-Bench (frozen, SHA-256 in the dataset repo).\n"

def card(kind):
    head = {"merged": f"Merged 16-bit weights (N-ATLaS + LoRA `{ADAPTER}`). Serve with vLLM or transformers.",
            "gguf": f"GGUF `{QUANT.upper()}` quantisation for llama.cpp / llama-cpp-python / LM Studio / Ollama (CPU friendly)."}[kind]
    return f"""---
base_model: NCAIR1/N-ATLaS
license: other
license_name: n-atlas-terms-of-use
language: [en, ha, yo]
tags: [n-atlas, health, primary-health-care, nigeria, hausa, yoruba{', gguf' if kind == 'gguf' else ''}]
---
# Ilera-ATLAS: Powered by Awarri

{head}

Ilera-ATLAS is **NCAIR1/N-ATLaS fine-tuned (QLoRA)** to give primary-health-care decision support that
follows Nigeria's **2024 National Standing Orders** for community health workers (CHPRBN, with FMoH and
NPHCDA), in **English, Hausa and Yoruba**. Every answer starts with an action and cites its source:

```
ACTION: TREAT | REFER | URGENT
WHAT TO DO: numbered steps
DANGER SIGNS: signs that mean urgent referral
SOURCE: document and section
```

## Intended use
Decision support for **trained health workers** (CHEWs, JCHEWs, CHOs). **Not** a diagnosis tool and
**not** for patients. Always apply clinical judgement; when in doubt, refer.

## Results
{results_table()}
## Use
Use the exact system prompt from `config/system_prompt.txt` in the `{DATA_REPO}` dataset; answers
follow the format only when it is present. Greedy decoding, `repetition_penalty=1.1`.

## Licence and attribution
Derived from N-ATLaS: released under the **N-ATLaS Terms of Use** (same licence for derivatives).
Attribution: *Awarri Technologies and the Federal Ministry of Communications, Innovation and Digital
Economy*. Use is capped at **1,000 active end-users per 30 days** without a commercial licence.
Prohibited uses of N-ATLaS apply (surveillance, profiling, disinformation, military use).
Guidance source: National Standing Orders 2024 (CHPRBN / FMoH / NPHCDA).

## Limitations
Yoruba quality is lower than English and Hausa (inherited from N-ATLaS). Medicines and doses must be
checked against the current Standing Orders. Code-switched and very long questions are less reliable.

_Exported {datetime.date.today()} by NAIC 2026 Ilera-ATLAS team._
"""

CARDS_DIR = os.path.abspath("cards")              # absolute: later cells change directory
os.makedirs(CARDS_DIR, exist_ok=True)
for kind in ("merged", "gguf"):
    open(os.path.join(CARDS_DIR, f"README_{kind}.md"), "w", encoding="utf-8").write(card(kind))
print(card("merged")[:1500])

## 2. Load the chosen adapter (N-ATLaS + LoRA)

In [ ]:
from unsloth import FastLanguageModel
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
model, tok = FastLanguageModel.from_pretrained(ADAPTER, max_seq_length=2048, load_in_4bit=True, token=HF_TOKEN)
os.chdir(WORK)                                    # large temporary files go to the big disk
for repo in (MERGED_REPO, GGUF_REPO):
    api.create_repo(repo, private=True, exist_ok=True)

## 3. Merged 16-bit weights → `ilera-atlas` (for vLLM on Modal)

In [ ]:
if DO_MERGED:
    model.push_to_hub_merged(MERGED_REPO, tok, save_method="merged_16bit", token=HF_TOKEN)
    print(f"merged: https://huggingface.co/{MERGED_REPO}")

## 4. GGUF `Q4_K_M` → `ilera-atlas-GGUF` (for the free CPU Space)
If this step fails (it compiles llama.cpp and needs disk), use the free [GGUF-my-repo](https://huggingface.co/spaces/ggml-org/gguf-my-repo) Space on the merged repo instead, choose `Q4_K_M`, then upload the card from `cards/README_gguf.md`.

In [ ]:
if DO_GGUF:
    model.push_to_hub_gguf(GGUF_REPO, tok, quantization_method=QUANT, token=HF_TOKEN)
    files = [f for f in api.list_repo_files(GGUF_REPO) if f.endswith(".gguf")]
    print("GGUF files:", files)
    assert files, "no .gguf file was uploaded: use the GGUF-my-repo fallback above"

In [ ]:
for repo, kind in ((MERGED_REPO, "merged"), (GGUF_REPO, "gguf")):
    api.upload_file(path_or_fileobj=os.path.join(CARDS_DIR, f"README_{kind}.md"), path_in_repo="README.md",
                    repo_id=repo, commit_message="Ilera-ATLAS model card")
print("model cards uploaded")

## 5. Sanity check the GGUF on CPU
Loads the quantised file exactly as the free Space will (llama-cpp-python, CPU only) and asks two
questions. Each answer must start with a valid `ACTION:` line. Expect a few tokens per second.

In [ ]:
if DO_SANITY:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python",
                    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cpu"], check=True)
    import re
    from llama_cpp import Llama
    from huggingface_hub import hf_hub_download
    SYSTEM = open(hf_hub_download(DATA_REPO, "config/system_prompt.txt", repo_type="dataset",
                                  token=HF_TOKEN), encoding="utf-8").read().strip()
    gguf = Llama.from_pretrained(GGUF_REPO, filename=f"*{QUANT.upper()}*.gguf", n_ctx=2048, n_threads=4,
                                 n_gpu_layers=0, verbose=False)
    for q in ["A 2-year-old has had fever for 3 days and is breathing fast. What should I do?",
              "Yaro mai shekara biyu yana da zazzabi da saurin numfashi. Me zan yi?"]:
        out = gguf.create_chat_completion(messages=[{"role": "system", "content": SYSTEM},
                                                    {"role": "user", "content": q}],
                                          temperature=0, max_tokens=250, repeat_penalty=1.1)["choices"][0]["message"]["content"]
        ok = bool(re.search(r"ACTION:\s*(TREAT|REFER|URGENT)", out))
        print(("OK  " if ok else "BAD ") + q + "\n" + out + "\n")

## Done when
* `ilera-atlas` (merged) and `ilera-atlas-GGUF` exist with model cards, and the GGUF sanity answers
  start with a valid `ACTION:` line.
* **Next (Fri 9 Oct):** `serve/` in the repo: `modal deploy serve/modal_app.py`, then
  `python serve/deploy_space.py` for the demo Space (runbook Part 11).